# Step 1: GRPO Training

Downloads SFT LoRA weights from Tinker → runs GRPO on GSM8K → saves to Drive.

**Runtime:** H100 GPU
**Secrets needed:** `TINKER_API_KEY`
**Time:** ~30 min per model × 3 models
**Output:** `grpo_lora_kimi/`, `grpo_lora_glm5/`, `grpo_lora_combined/` on Drive

In [ ]:
# Install — use Unsloth's official install script (handles transformers v5 for Qwen3.5)
# Standard pip install does NOT work on Colab — pins transformers<=4.57.6 which is too old
# Restart runtime after this cell!
!curl -fsSL https://unsloth.ai/install.sh | sh
!pip install tinker lm-eval --quiet
print('Done. Now: Runtime → Restart session, then run the next cell.')

In [ ]:
# Setup (run after restart)
import transformers
assert hasattr(transformers, '__version__')
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert 'qwen3_5' in CONFIG_MAPPING, f'Need transformers>=5.0, got {transformers.__version__}'
print(f'transformers {transformers.__version__} — qwen3_5 ✅')

from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json
DRIVE = '/content/drive/MyDrive/distillreasoning'
os.makedirs(DRIVE, exist_ok=True)
print(f'Drive: {DRIVE} ✅')

In [ ]:
# Download LoRA weights from Tinker + fix adapter config
import tinker, requests, tarfile

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
service = tinker.ServiceClient()
rest = service.create_rest_client()

BASE_MODEL = 'Qwen/Qwen3.5-4B'

SFT_CHECKPOINTS = {
    'glm5': 'tinker://0fbca836-2aae-5500-b28d-93c2a46a328b:train:0/sampler_weights/qwen35-4b-glm5-final',
    'kimi': 'tinker://f5795e66-71e4-5cf4-9ebe-1cc14c27aa6e:train:0/sampler_weights/qwen35-4b-kimi-final',
    'combined': 'tinker://41e7bd9e-e49f-5f13-a5ff-f4339faab448:train:0/sampler_weights/qwen35-4b-combined-final',
}

for name, tinker_path in SFT_CHECKPOINTS.items():
    lora_dir = f'{DRIVE}/sft_lora_{name}'
    if os.path.exists(lora_dir) and any(f.endswith('.safetensors') for f in os.listdir(lora_dir)):
        print(f'{name}: already downloaded')
    else:
        print(f'Downloading {name}...')
        url_resp = rest.get_checkpoint_archive_url_from_tinker_path(tinker_path).result()
        local_tar = f'/tmp/lora_{name}.tar'
        r = requests.get(url_resp.url, stream=True)
        with open(local_tar, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        os.makedirs(lora_dir, exist_ok=True)
        with tarfile.open(local_tar) as t:
            t.extractall(lora_dir)
        print(f'  Downloaded: {os.listdir(lora_dir)}')
    
    # Fix adapter_config — Tinker leaves base_model_name_or_path as null
    config_path = os.path.join(lora_dir, 'adapter_config.json')
    if os.path.exists(config_path):
        config = json.load(open(config_path))
        if config.get('base_model_name_or_path') is None:
            config['base_model_name_or_path'] = BASE_MODEL
            json.dump(config, open(config_path, 'w'), indent=2)
            print(f'  Patched adapter_config: base_model={BASE_MODEL}')

print('\nAll LoRA weights ready!')

In [ ]:
# GRPO setup — dataset + reward functions
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset
import torch, re

max_seq_length = 2048
lora_rank = 32

SYSTEM_PROMPT = """Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>"""

def extract_hash_answer(text):
    if '####' not in text: return None
    return text.split('####')[1].strip()

def get_gsm8k_questions(split='train'):
    data = load_dataset('openai/gsm8k', 'main')[split]
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']},
        ],
        'answer': extract_hash_answer(x['answer']),
    })
    return data

dataset = get_gsm8k_questions()
print(f'GRPO dataset: {len(dataset)} GSM8K train problems')

def extract_xml_answer(text):
    if '<answer>' in text and '</answer>' in text:
        return text.split('<answer>')[-1].split('</answer>')[0].strip()
    numbers = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return numbers[-1] if numbers else ''

def correctness_reward(prompts, completions, answer, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    return [2.0 if e == a else 0.0 for e, a in zip(extracted, answer)]

def format_reward(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    return [0.5 if '<reasoning>' in r and '</reasoning>' in r and '<answer>' in r else 0.0 for r in responses]

def int_reward(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.replace('-','').replace('.','').isdigit() else 0.0 for r in extracted]

print('Reward functions ready ✅')

In [ ]:
# Run GRPO for each SFT model (saves to Drive after each)
for teacher in ['kimi', 'glm5', 'combined']:
    grpo_dir = f'{DRIVE}/grpo_lora_{teacher}'
    if os.path.exists(grpo_dir) and os.listdir(grpo_dir):
        print(f'{teacher}: GRPO already done, skipping')
        continue
    
    print(f'\n{"="*60}')
    print(f'GRPO: {teacher}')
    print(f'{"="*60}')
    
    sft_dir = f'{DRIVE}/sft_lora_{teacher}'
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=sft_dir,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        fast_inference=True,
        max_lora_rank=lora_rank,
        gpu_memory_utilization=0.6,
    )
    
    training_args = GRPOConfig(
        learning_rate=5e-6,
        adam_beta1=0.9,
        adam_beta2=0.99,
        weight_decay=0.1,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        optim='paged_adamw_8bit',
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=8,
        max_prompt_length=512,
        max_completion_length=max_seq_length - 512,
        max_steps=250,
        save_steps=250,
        max_grad_norm=0.1,
        report_to='none',
        output_dir=f'/tmp/grpo_{teacher}',
    )
    
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[correctness_reward, format_reward, int_reward],
        args=training_args,
        train_dataset=dataset,
    )
    
    print(f'  Training (250 steps)...')
    trainer.train()
    
    model.save_lora(grpo_dir)
    print(f'  ✅ Saved: {grpo_dir}')
    
    del model, tokenizer, trainer
    torch.cuda.empty_cache()

print('\n🎉 All GRPO training complete! Run 02_eval.ipynb next.')